# M3: Combined Model — M1 Futures Adjustment + M2 Jump Diffusion

This notebook implements the combined extension M3, which applies both:
- **M1**: Oil futures term structure adjustment to the junior claim (LCL)
- **M2**: OVX-calibrated jump-diffusion asset process

The combination is not a linear superposition. M1 modifies the balance sheet input
(LCL_aug = LCL_usd × futures_term), and M2 modifies the asset dynamics (jump-diffusion
pricing). Running both simultaneously means the M1-adjusted LCL is fed into the
jump-diffusion solver — the nonlinear inversion produces a genuinely different
implied asset value and volatility from either extension alone.

The combined model is estimated with identical parameters to M1 and M2:
- eta = 1 (futures term scaling, M1)
- a_fit, b_fit, lam_max, eta_jd (OVX logistic + jump size, M2)
This preserves the quasi-experimental identification — no country-specific tuning.

In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from CCA_utils import *

## 1. Load Base Panel

In [2]:
study_sovereigns = [
    'Saudi Arabia', 'Abu Dhabi', 'Dubai', 'Qatar', 'Colombia',
    'Mexico', 'Brazil', 'Egypt', 'Malaysia', 'Indonesia',
    'Philippines', 'Turkey', 'Chile', 'China',
    'South Africa', 'South Korea', 'Thailand'
]

cca_panel_df = pd.read_csv('../data/processed/CCA/cca_newfx_rates.csv')
cca_panel_df['date'] = pd.to_datetime(cca_panel_df['date'])
cca_panel_df = cca_panel_df[cca_panel_df['country'].isin(study_sovereigns)].copy()
cca_panel_df.drop(columns=['cds_spread_1Y'], inplace=True)
cca_panel_df.rename(columns={'cds_spread_5Y': 'cds_spread'}, inplace=True)

T          = 1.0
vol_window = 52
freq       = 'W'

cca_panel_df['domestic_rate_in_units']    = cca_panel_df['domestic_rate_in_units'] / 100
cca_panel_df['risk_free_rate']            = cca_panel_df['risk_free_rate'] / 100
cca_panel_df['monetary_base_mn_localcurr'] = cca_panel_df['monetary_base_mn_localcurr'] / 1000
cca_panel_df['domestic_debt_bn_localcurr'] = cca_panel_df['domestic_debt_bn_localcurr']
cca_panel_df['external_debt_mn_usd']      = cca_panel_df['external_debt_mn_usd'] / 1000

## 2. Load M1 Data (Oil Futures Term Structure)

In [3]:
# Oil spot and 12-month futures — identical to M1 notebook
oil_prices  = pd.read_csv('../data/processed/Oil/oil_prices_datastream.csv')
oil_prices['date'] = pd.to_datetime(oil_prices['date'])
oil_futures = pd.read_csv('../data/processed/Oil/oil_futures.csv')
oil_futures['date'] = pd.to_datetime(oil_futures['date'])

cca_panel_df = cca_panel_df.merge(oil_prices[['date', 'Brent']], on='date', how='left')
cca_panel_df = cca_panel_df.merge(oil_futures[['date', 'Brent_12m']], on='date', how='left')

# Oil production — UAE mapped to Abu Dhabi and Dubai
oil_prod = pd.read_csv('../data/processed/Oil/crude_oil_production.csv')
oil_prod = oil_prod.rename(columns={oil_prod.columns[0]: 'country'})
oil_prod = oil_prod.melt(
    id_vars='country', var_name='year', value_name='oil_production_Mt_yr'
)
oil_prod['year'] = oil_prod['year'].astype(int)

uae = oil_prod[oil_prod['country'] == 'United Arab Emirates'].copy()
for name in ['Abu Dhabi', 'Dubai']:
    rows = uae.copy()
    rows['country'] = name
    oil_prod = pd.concat([oil_prod, rows], ignore_index=True)

cca_panel_df['year'] = cca_panel_df['date'].dt.year
cca_panel_df = cca_panel_df.merge(
    oil_prod[['year', 'country', 'oil_production_Mt_yr']],
    on=['country', 'year'], how='left'
)
cca_panel_df['oil_production_Mt_yr'] = cca_panel_df['oil_production_Mt_yr'].fillna(0).astype(int)

C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_43052\1290904706.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  oil_prices['date'] = pd.to_datetime(oil_prices['date'])
C:\Users\JuanFranciscoPerez\AppData\Local\Temp\ipykernel_43052\1290904706.py:5: UserWarning: Parsing dates in %d.%m.%Y format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  oil_futures['date'] = pd.to_datetime(oil_futures['date'])


## 3. Load M2 Data (OVX)

In [4]:
# OVX — identical to M2 notebook
ovx_df = pd.read_csv('../data/processed/Macroeconomic_variables/OVXCLS.csv')
ovx_df['date'] = pd.to_datetime(ovx_df['date'])
cca_panel_df = cca_panel_df.merge(ovx_df[['date', 'OVXCLS']], on='date', how='left')

# Resample to weekly
cca_panel_df.set_index(['date', 'country'], inplace=True)
cca_panel_df = (
    cca_panel_df
    .groupby('country')
    .resample(freq, level='date')
    .last()
)
cca_panel_df.reset_index(inplace=True)
cca_panel_df['OVXCLS'] = cca_panel_df.groupby('country')['OVXCLS'].ffill()

print(f"OVX missing values: {cca_panel_df['OVXCLS'].isna().sum()}")
print(f"Panel shape: {cca_panel_df.shape}")

OVX missing values: 0
Panel shape: (9758, 19)


## 4. Model Parameters

All parameters identical to M1 and M2 — no re-estimation.

In [5]:
# --- M1 parameter ---
eta_futures = 1       # futures term scaling: (spot/12m_futures)^eta

# --- M2 parameters (identical to M2 notebook) ---
a_fit   =  0.0780     # logistic slope on OVX
b_fit   = -4.0223     # logistic intercept
lam_max =  2.0        # upper bound on annualised jump intensity
eta_jd  =  0.0806     # single-parameter jump: mu_J = -eta_jd, sigma_J = eta_jd

def ovx_to_jump_params(ovx):
    """
    Maps OVX to annualised Poisson intensity via logistic mapping.
    Returns (lambda, mu_J, sigma_J) under single-parameter constraint.
    Identical to M2 notebook.
    """
    if np.isnan(ovx):
        return 0.0, 0.0, 0.0
    lam = lam_max / (1.0 + np.exp(-b_fit - a_fit * ovx))
    return lam, -eta_jd, eta_jd

# Jump-diffusion pricer — identical to M2 notebook
pricer = AnalyticalJumpDiffusionPricer(max_jumps=20)

## 5. M3 Calibration Loop

The only difference from M1 and M2:
- M1 feeds `LCL_aug` into `solve_CCA` (GBM pricer)
- M2 feeds `LCL_usd` into `pricer.solve_CCA_jd` (jump-diffusion pricer)
- **M3 feeds `LCL_aug` into `pricer.solve_CCA_jd`** — both adjustments active simultaneously

In [6]:
results = pd.DataFrame()

print("Starting M3 Combined CCA Calibration loop...")

for country, group in cca_panel_df.groupby('country'):
    print(f"Processing {country}...")
    df = group.copy().sort_values('date').reset_index(drop=True)

    r_d      = df['domestic_rate_in_units']
    r_f      = df['risk_free_rate']
    M_bn     = df['monetary_base_mn_localcurr']
    dom_D_bn = df['domestic_debt_bn_localcurr']
    ext_D_bn = df['external_debt_mn_usd']
    fx_rate  = df['fx_rate']

    # ── Step 1: Compute base LCL (same as M0 and M2) ──────────
    df['LCL_usd'] = [
        compute_lcl_usd(m, bd, fx, rd, rf, T)
        for m, bd, fx, rd, rf in zip(M_bn, dom_D_bn, fx_rate, r_d, r_f)
    ]

    # ── Step 2: Apply M1 futures adjustment (same as M1) ──────
    # futures_term = (spot / 12m_futures)^eta_futures
    # Contango  (spot < futures) → term < 1 → LCL_aug < LCL_usd → tighter fiscal
    # Backwardation (spot > futures) → term > 1 → LCL_aug > LCL_usd → looser fiscal
    df['futures_term'] = (df['Brent'] / df['Brent_12m']) ** eta_futures
    df['LCL_aug']      = df['LCL_usd'] * df['futures_term']

    # ── Step 3: Compute volatility on adjusted LCL (same as M1) 
    ann_factor    = np.sqrt(52) if freq == 'W' else np.sqrt(12)
    log_ret       = np.log(df['LCL_aug'] / df['LCL_aug'].shift(1))
    df['sigma_lcl'] = log_ret.rolling(window=vol_window).std() * ann_factor

    # ── Step 4: Compute distress barrier (same as M0/M1/M2) ───
    df['B_f'] = [
        compute_barrier_kvm(debt, rf, T)
        for debt, rf in zip(ext_D_bn, r_f)
    ]

    # ── Step 5: Solve CCA with M1-adjusted LCL + M2 jump process
    out = {
        'implied_V': [], 'implied_sigma_V': [], 'cca_converged': [],
        'distance_to_distress': [], 'default_prob': [],
        'model_spread_bps': [], 'put_value': [],
        'risky_debt': [], 'leverage': []
    }

    for i, row in df.iterrows():

        # OVX → jump parameters (identical to M2)
        lam_ovx_ann, mu_ovx, sigma_ovx = ovx_to_jump_params(row['OVXCLS'])

        # Warm-start from GBM solver on M1-adjusted LCL
        # This is the key difference from M2: warm start uses LCL_aug not LCL_usd
        cca_base = solve_CCA(
            row['LCL_aug'], row['sigma_lcl'], row['B_f'], r_f.iloc[i], T
        )
        v_guess   = cca_base['V']       if cca_base['converged'] else (row['LCL_aug'] + row['B_f'])
        sig_guess = cca_base['sigma_V'] if cca_base['converged'] else (
            row['sigma_lcl'] * row['LCL_aug'] / v_guess
        )

        # Solve CCA with M1-adjusted LCL fed into M2 jump-diffusion pricer
        cca_jd = pricer.solve_CCA_jd(
            LCL_usd   = row['LCL_aug'],   # <-- M1 adjustment applied here
            sigma_lcl = row['sigma_lcl'],
            B_f       = row['B_f'],
            r_f       = r_f.iloc[i],
            T         = T,
            lam_base  = 0,
            mu_base   = 0,
            sig_base  = 0,
            lam_ovx   = lam_ovx_ann,
            mu_ovx    = mu_ovx,
            sig_ovx   = sigma_ovx,
            v_guess   = v_guess,
            sig_guess = sig_guess
        )

        if cca_jd['converged']:
            risk = pricer.compute_risk_jd(
                V          = cca_jd['V'],
                sigma_diff = cca_jd['sigma_diff'],
                B_f        = row['B_f'],
                r_f        = r_f.iloc[i],
                T          = T,
                lam_base   = 0,
                mu_base    = 0,
                sig_base   = 0,
                lam_ovx    = lam_ovx_ann,
                mu_ovx     = mu_ovx,
                sig_ovx    = sigma_ovx
            )
            out['implied_V'].append(cca_jd['V'])
            out['implied_sigma_V'].append(cca_jd['sigma_total'])
            out['cca_converged'].append(True)
            out['distance_to_distress'].append(risk.get('d2', np.nan))
            out['default_prob'].append(risk.get('default_prob', np.nan))
            out['model_spread_bps'].append(risk.get('credit_spread_bps', np.nan))
            out['put_value'].append(risk.get('put_value', np.nan))
            out['risky_debt'].append(risk.get('risky_debt', np.nan))
            out['leverage'].append(risk.get('leverage', np.nan))
        else:
            for key in out:
                out[key].append(np.nan)
            out['cca_converged'][-1] = False

    for col, vals in out.items():
        df[col] = vals

    results = pd.concat([results, df])

print("Calibration complete!")

Starting M3 Combined CCA Calibration loop...
Processing Abu Dhabi...
Processing Brazil...
Processing Chile...
Processing China...
Processing Colombia...
Processing Dubai...
Processing Egypt...
Processing Indonesia...
Processing Malaysia...
Processing Mexico...
Processing Philippines...
Processing Qatar...
Processing Saudi Arabia...
Processing South Africa...
Processing South Korea...
Processing Thailand...
Processing Turkey...
Calibration complete!


## 6. Filter to Sample Period and Save

In [7]:
START_DATE = '2015-01-01'
END_DATE   = '2024-12-31'

results = results[
    (results['date'] >= START_DATE) & (results['date'] <= END_DATE)
].copy()

print(f"Final results shape: {results.shape}")
print(f"Countries: {sorted(results['country'].unique())}")
print(f"Date range: {results['date'].min()} to {results['date'].max()}")
print(f"Convergence rate: {results['cca_converged'].mean():.1%}")

results.to_csv('../output/results/M3_results.csv', index=False)
print("Saved: ../output/results/M3_results.csv")

Final results shape: (8874, 33)
Countries: ['Abu Dhabi', 'Brazil', 'Chile', 'China', 'Colombia', 'Dubai', 'Egypt', 'Indonesia', 'Malaysia', 'Mexico', 'Philippines', 'Qatar', 'Saudi Arabia', 'South Africa', 'South Korea', 'Thailand', 'Turkey']
Date range: 2015-01-04 00:00:00 to 2024-12-29 00:00:00
Convergence rate: 100.0%
Saved: ../output/results/M3_results.csv


## 7. Quick Sanity Check

Compare average DD across M0, M1, M2, M3 to confirm M3 is not identical to either extension alone.

In [8]:
m0 = pd.read_csv('../output/results/M0_results.csv', parse_dates=['date'])
m1 = pd.read_csv('../output/results/M1_results.csv', parse_dates=['date'])
m2 = pd.read_csv('../output/results/M2_results.csv', parse_dates=['date'])
m3 = results.copy()

print("Average DD across full sample:")
print(f"  M0: {m0['distance_to_distress'].mean():.3f}")
print(f"  M1: {m1['distance_to_distress'].mean():.3f}")
print(f"  M2: {m2['distance_to_distress'].mean():.3f}")
print(f"  M3: {m3['distance_to_distress'].mean():.3f}")
print()
print("If M3 = M1 or M3 = M2 exactly, there is a bug.")
print("M3 should sit between M1 and M2 or below both.")

Average DD across full sample:
  M0: 19.185
  M1: 10.629
  M2: 15.858
  M3: 10.558

If M3 = M1 or M3 = M2 exactly, there is a bug.
M3 should sit between M1 and M2 or below both.
